# Experimentos com outras variáveis

Existe um limite a árvore e o Naive Bayes a 4–6 atributos e a rede bayesiana a 4–6 nós, e o trabalho já usa 6. Por isso cada variável candidata é testada **trocando** uma variável da rede (ou, quando indicado, como 7º nó), sem mudar o pipeline principal.

Para cada configuração da rede da Etapa 5 medimos, com probabilidades fora da dobra:
- **AUC 2024:** validação cruzada estratificada de 5 dobras (tercis das taxas calculados no treino);
- **AUC estado de fora:** treina com 8 estados e testa no 9º;
- **AUC entrada 2025:** entre os 549 municípios sem facção em 2024, quais passaram a ter em 2025.

Também testamos uma regressão logística só para a entrada em 2025, que aproveita melhor uma variável nova.

As bases externas são montadas por `montar_bases_externas.py` (IBGE, INPE e IBAMA).

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from utils import montar_tabela, juntar_externas, tercis_do_treino, SEED
np.random.seed(SEED)
d = juntar_externas(montar_tabela())
y, y25 = d["ALVO"].values, d["ALVO_2025"].values
sem = y == 0
dobras = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(d, y))
por_uf = [(np.where(d["UF"] != u)[0], np.where(d["UF"] == u)[0]) for u in sorted(d["UF"].unique())]
print(d.shape)

(808, 129)


In [2]:
CONTINUAS = {}   # nome do nó -> coluna numérica discretizada em tercis com os cortes do treino

def p_fora(pais, filhos, divisoes):
    """P(FACCAO=1) de cada município pela rede PAIS -> FACCAO -> FILHOS treinada sem ele."""
    ev = pais + filhos
    arestas = [(p, "FACCAO") for p in pais] + [("FACCAO", f) for f in filhos]
    X = d[ev].copy(); X["FACCAO"] = y
    p = np.full(len(d), np.nan)
    for tr, te in divisoes:
        Xf = tercis_do_treino(d, X, tr)
        for no, col in CONTINUAS.items():
            if no in Xf:
                Xf[no] = np.digitize(d[col].values, np.quantile(d[col].values[tr], [1 / 3, 2 / 3]), right=True)
        m = DiscreteBayesianNetwork(arestas); m.fit(Xf.iloc[tr].astype(int))
        inf, cache = VariableElimination(m), {}
        for i, r in zip(te, Xf.iloc[te][ev].values.astype(int)):
            k = tuple(int(v) for v in r)
            if k not in cache:
                q = inf.query(["FACCAO"], evidence=dict(zip(ev, k)), show_progress=False)
                cache[k] = float(q.values[list(q.state_names["FACCAO"]).index(1)])
            p[i] = cache[k]
    return p

resultados = []
def avaliar(nome, pais, filhos, grupo):
    p, pu = p_fora(pais, filhos, dobras), p_fora(pais, filhos, por_uf)
    r = {"grupo": grupo, "configuração": nome,
         "AUC 2024": np.mean([roc_auc_score(y[te], p[te]) for _, te in dobras]),
         "AUC estado de fora": roc_auc_score(y, pu),
         "AUC entrada 2025": roc_auc_score(y25[sem], p[sem])}
    resultados.append(r)
    return pd.Series({k: v for k, v in r.items() if k.startswith("AUC")}), pu

ATUAL = (["PORTE", "FRONT"], ["TXH", "ESC", "TXT"])
avaliar("rede atual (Etapa 5)", *ATUAL, "referência")[0]

AUC 2024              0.809527
AUC estado de fora    0.799530
AUC entrada 2025      0.763944
dtype: float64

## 1. Fronteira por município em vez de por estado

`FRONT` é definido por estado e só separa MA e TO dos demais; por isso a rede erra o nível de facção no Acre e em Roraima (Etapa 5). Testamos duas versões municipais, da lista oficial do IBGE (2024): o município está na **faixa de fronteira** (150 km da divisa) ou **encosta na divisa**.

In [3]:
d["FRONT_FAIXA"] = d["faixa_fronteira"].astype(int)
d["FRONT_LIMITE"] = d["toca_limite"].astype(int)
print(d.groupby("UF")[["FRONT", "FRONT_FAIXA", "FRONT_LIMITE"]].sum().T, "\n")
niveis = {}
for nome, col in [("FRONT por estado", "FRONT"), ("FRONT = faixa de fronteira", "FRONT_FAIXA"), ("FRONT = encosta na divisa", "FRONT_LIMITE")]:
    r, pu = avaliar(nome, ["PORTE", col], ATUAL[1], "fronteira")
    print(nome, r.round(3).to_dict())
    niveis[nome] = pd.Series(pu).groupby(d["UF"].values).mean()
print("\nP(facção) média prevista com o estado de fora:")
pd.DataFrame(niveis).assign(**{"facção real": d.groupby("UF")["ALVO"].mean()}).round(3).T

UF            AC  AM  AP  MA   MT   PA  RO  RR  TO
FRONT         22  62  16   0  141  144  52  15   0
FRONT_FAIXA   22  19   8   0   28    5  28  15   0
FRONT_LIMITE  17   8   2   0    4    3   9   9   0 



FRONT por estado {'AUC 2024': 0.81, 'AUC estado de fora': 0.8, 'AUC entrada 2025': 0.764}


FRONT = faixa de fronteira {'AUC 2024': 0.837, 'AUC estado de fora': 0.811, 'AUC entrada 2025': 0.708}


FRONT = encosta na divisa {'AUC 2024': 0.824, 'AUC estado de fora': 0.802, 'AUC entrada 2025': 0.705}

P(facção) média prevista com o estado de fora:


,AC,AM,AP,MA,MT,PA,RO,RR,TO
FRONT por estado,0.465,0.324,0.507,0.205,0.507,0.528,0.427,0.469,0.110
FRONT = faixa de fronteira,0.681,0.337,0.561,0.281,0.348,0.373,0.466,0.701,0.183
FRONT = encosta na divisa,0.786,0.302,0.437,0.313,0.311,0.402,0.411,0.699,0.235
facção real,1.000,0.323,0.312,0.221,0.298,0.507,0.500,0.933,0.065


In [4]:
s = d[sem & (d["FRONT"] == 1).values]
print("Entrada em 2025 nos estados de fronteira, entre os sem facção em 2024:")
s.groupby("FRONT_FAIXA")["ALVO_2025"].agg(["size", "mean"]).rename(index={0: "interior", 1: "faixa de fronteira"}).round(3)

Entrada em 2025 nos estados de fronteira, entre os sem facção em 2024:


,size,mean
FRONT_FAIXA,,
interior,208,0.385
faixa de fronteira,42,0.357


**Leitura.** A fronteira por município melhora o diagnóstico de 2024 (AUC 0,81 → 0,84) e corrige boa parte do erro de nível no Acre e em Roraima. Mas piora a previsão da entrada em 2025 (0,76 → 0,71): a expansão de 2025 aconteceu tanto no interior dos estados de fronteira quanto na faixa, e a variável municipal rebaixa justamente o interior do Pará, do Mato Grosso e do Amazonas. Como o objetivo final do pipeline é antecipar a expansão, mantivemos `FRONT` por estado.

## 2. Outras variáveis do VDE e variáveis "placebo"

Taxa anual 2021–23 por 100 mil habitantes. Variáveis raras (presentes em menos da metade dos municípios) viram "teve/não teve"; as demais, tercis. Suicídio e morte no trânsito funcionam como **placebo**: não têm relação direta com facções.

In [5]:
vde = {"MANDADOS": "mandados", "FEMINIC": "feminicidio", "LESAO_MORTE": "lesao_morte", "LATROC": "latrocinio",
       "TRANSITO": "morte_transito", "SUICIDIO": "suicidio"}
for no, ev in vde.items():
    taxa = sum(d[f"{ev}_{a}"] for a in (2021, 2022, 2023)) / 3 / d["pop_2022"] * 1e5
    if (taxa > 0).mean() > 0.5:
        d[f"taxa_{ev}_x"] = taxa; CONTINUAS[no] = f"taxa_{ev}_x"; d[no] = 0
    else:
        d[no] = (taxa > 0).astype(int)
    avaliar(f"{no} no lugar de ESC", ATUAL[0], ["TXH", no, "TXT"], "VDE")
    avaliar(f"{no} como 7º nó", ATUAL[0], ATUAL[1] + [no], "VDE")
pd.DataFrame([r for r in resultados if r["grupo"] == "VDE"]).drop(columns="grupo").round(3)

,configuração,AUC 2024,AUC estado de fora,AUC entrada 2025
0,MANDADOS no lugar de ESC,0.812,0.798,0.757
1,MANDADOS como 7º nó,0.809,0.795,0.768
2,FEMINIC no lugar de ESC,0.811,0.799,0.738
3,FEMINIC como 7º nó,0.808,0.800,0.753
4,LESAO_MORTE no lugar de ESC,0.812,0.800,0.742
5,LESAO_MORTE como 7º nó,0.807,0.799,0.758
6,LATROC no lugar de ESC,0.819,0.807,0.732
7,LATROC como 7º nó,0.816,0.806,0.750
8,TRANSITO no lugar de ESC,0.815,0.799,0.752
9,TRANSITO como 7º nó,0.811,0.797,0.768


**Leitura.** Nenhuma troca muda a AUC de 2024 em mais de 0,01, e todas pioram a entrada em 2025. Suicídio e morte no trânsito rendem tanto quanto as variáveis de violência: os "sintomas" da rede carregam sobretudo informação sobre o tamanho e a urbanização do município.

## 3. IBGE: vizinhos com facção, urbanização e hierarquia urbana

- **VIZ:** algum município vizinho (malha do IBGE) tinha facção em 2024 — hipótese de contágio espacial;
- **URB:** taxa de urbanização (Censo 2022), em tercis;
- **REGIC:** hierarquia urbana (REGIC 2018): centro local, centro de zona, ou cidade mais central.

In [6]:
fac = dict(zip(d["cod_ibge"], d["ALVO"]))
d["VIZ"] = [int(any(fac.get(v, 0) for v in str(vs).split(";"))) for vs in d["vizinhos"]]
CONTINUAS["URB"] = "taxa_urbanizacao"; d["URB"] = 0
d["REGIC"] = d["regic_nivel"].map(lambda n: 0 if n == 5 else (1 if n == 4 else 2))
d["URB_T"] = pd.qcut(d["taxa_urbanizacao"].rank(method="first"), 3, labels=False)
for no in ["VIZ", "URB_T", "REGIC"]:
    print(f"{no}: P(facção 2024) = {d.groupby(no)['ALVO'].mean().round(3).to_dict()}"
          f" | entrada 2025 = {d[sem].groupby(no)['ALVO_2025'].mean().round(3).to_dict()}")
for no in ["VIZ", "URB", "REGIC"]:
    avaliar(f"{no} no lugar de ESC", ATUAL[0] + [no], ["TXH", "TXT"], "IBGE")
avaliar("VIZ no lugar de FRONT", ["PORTE", "VIZ"], ATUAL[1], "IBGE")
pd.DataFrame([r for r in resultados if r["grupo"] == "IBGE"]).drop(columns="grupo").round(3)

VIZ: P(facção 2024) = {0: 0.086, 1: 0.391} | entrada 2025 = {0: 0.129, 1: 0.283}
URB_T: P(facção 2024) = {0: 0.274, 1: 0.275, 2: 0.413} | entrada 2025 = {0: 0.148, 1: 0.272, 2: 0.297}
REGIC: P(facção 2024) = {0: 0.259, 1: 0.315, 2: 0.807} | entrada 2025 = {0: 0.226, 1: 0.28, 2: 0.375}


,configuração,AUC 2024,AUC estado de fora,AUC entrada 2025
0,VIZ no lugar de ESC,0.819,0.800,0.742
1,URB no lugar de ESC,0.819,0.788,0.731
2,REGIC no lugar de ESC,0.807,0.787,0.751
3,VIZ no lugar de FRONT,0.799,0.783,0.707


**Leitura.** Sozinhas, as três se associam à facção, mas na rede não acrescentam nada: o contágio entre vizinhos já está capturado pelo estado, e urbanização e centralidade são outras medidas do tamanho do município.

## 4. INPE e IBAMA: garimpo, desmatamento e fiscalização

- **GARIMPO_IBAMA / GARIMPO_INPE:** ver Etapa 3, seção 8;
- **DESMAT:** área desmatada 2021–23 segundo o DETER, em tercis;
- **DESTRUICAO:** o IBAMA destruiu equipamentos de infratores no município (termo de destruição).

In [7]:
CONTINUAS["DESMAT"] = "deter_desmat_km2"; d["DESMAT"] = 0
d["DESTRUICAO"] = (d["ibama_termos_destruicao"] > 0).astype(int)
for no in ["GARIMPO_IBAMA", "GARIMPO_INPE", "DESMAT", "DESTRUICAO"]:
    avaliar(f"{no} no lugar de ESC", ATUAL[0] + [no], ["TXH", "TXT"], "INPE/IBAMA")
pd.DataFrame([r for r in resultados if r["grupo"] == "INPE/IBAMA"]).drop(columns="grupo").round(3)

,configuração,AUC 2024,AUC estado de fora,AUC entrada 2025
0,GARIMPO_IBAMA no lugar de ESC,0.813,0.794,0.747
1,GARIMPO_INPE no lugar de ESC,0.809,0.797,0.746
2,DESMAT no lugar de ESC,0.808,0.796,0.733
3,DESTRUICAO no lugar de ESC,0.809,0.786,0.749


## 5. Regressão logística só para a entrada em 2025

Entre os 549 sem facção em 2024. Base: log(população), log das taxas de homicídio e tentativa, proporção de municípios com facção no estado e `FRONT`. Validação cruzada de 5 dobras.

In [8]:
s = d[sem].copy()
s["share_uf"] = s["UF"].map(d.groupby("UF")["ALVO"].mean())
s["lpop"] = np.log(s["pop_2022"])
s["lth"], s["ltt"] = np.log1p(s["taxa_homicidio_2021_23"]), np.log1p(s["taxa_tent_homicidio_2021_23"])
for c in ["deter_desmat_km2", "ibama_autos_garimpo", "ibama_termos_destruicao", "ibama_autos_flora"]:
    s["log_" + c] = np.log1p(s[c])
s["VIZ_FR"] = s["vizinhos"].map(lambda vs: np.mean([fac.get(v, 0) for v in str(vs).split(";")]))
base = ["lpop", "lth", "ltt", "share_uf", "FRONT"]
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
linhas = []
for extra in [[], ["VIZ_FR"], ["taxa_urbanizacao"], ["regic_nivel"], ["GARIMPO_IBAMA"], ["GARIMPO_INPE"],
              ["log_deter_desmat_km2"], ["log_ibama_termos_destruicao"],
              ["GARIMPO_IBAMA", "log_deter_desmat_km2", "log_ibama_termos_destruicao"]]:
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    a = cross_val_score(m, s[base + extra], s["ALVO_2025"], cv=cv, scoring="roc_auc")
    linhas.append({"atributos": "base" + "".join(" + " + e for e in extra), "AUC": a.mean(), "desvio entre dobras": a.std()})
pd.DataFrame(linhas).round(3)

,atributos,AUC,desvio entre dobras
0,base,0.779,0.052
1,base + VIZ_FR,0.774,0.051
2,base + taxa_urbanizacao,0.779,0.051
3,base + regic_nivel,0.769,0.040
4,base + GARIMPO_IBAMA,0.779,0.053
5,base + GARIMPO_INPE,0.779,0.052
6,base + log_deter_desmat_km2,0.786,0.056
7,base + log_ibama_termos_destruicao,0.781,0.057
8,base + GARIMPO_IBAMA + log_deter_desmat_km2 + ...,0.789,0.058


## 6. Resumo

In [9]:
pd.DataFrame(resultados).round(3)

,grupo,configuração,AUC 2024,AUC estado de fora,AUC entrada 2025
0,referência,rede atual (Etapa 5),0.810,0.800,0.764
1,fronteira,FRONT por estado,0.810,0.800,0.764
2,fronteira,FRONT = faixa de fronteira,0.837,0.811,0.708
3,fronteira,FRONT = encosta na divisa,0.824,0.802,0.705
4,VDE,MANDADOS no lugar de ESC,0.812,0.798,0.757
5,VDE,MANDADOS como 7º nó,0.809,0.795,0.768
6,VDE,FEMINIC no lugar de ESC,0.811,0.799,0.738
7,VDE,FEMINIC como 7º nó,0.808,0.800,0.753
8,VDE,LESAO_MORTE no lugar de ESC,0.812,0.800,0.742
9,VDE,LESAO_MORTE como 7º nó,0.807,0.799,0.758


**Conclusão.** Nenhuma das variáveis testadas melhora de forma consistente o diagnóstico de 2024 e a previsão da entrada em 2025 ao mesmo tempo. As diferenças na logística ficam dentro da variação entre dobras. Porte e estado explicam a maior parte da presença registrada de facção, e as demais variáveis repetem essa informação. O garimpo é a exceção qualitativa: dobra a taxa de entrada em 2025 (Etapa 3), mas atinge poucos municípios para mudar a AUC. O que limita os resultados parece ser o rótulo ("sem registro" não é ausência), e não a falta de atributos.